In [ ]:
import torch
import plotly.express

In [ ]:
for dim in [100, 10000]:
    for vocab_size in [10000, 100000]:
        torch.manual_seed(1)
        embs = torch.normal(mean=0, std=1, size=(vocab_size, dim))
        idx_1, idx_2 = 1000, 2000
        logits = (0.5 * embs[idx_1] + 0.5 * embs[idx_2]) @ embs.T
        probs = logits.softmax(dim=-1)
        probs, idcs = probs.topk(10, sorted=True)
        plotly.express.bar(y=probs, x=list(map(str, idcs.tolist())), title=f"Random Embs top10 (vocab={vocab_size}, dim={dim})", text_auto=".6f").show()

In [ ]:
import itertools
import tqdm
import polars as pl
df = []
dims = [50, 100, 500, 1000, 5000, 10000]
vocab = [1000, 5000, 10000, 50000, 100000]

for dim, vocab_size in tqdm.tqdm(itertools.product(dims, vocab), total=len(dims) * len(vocab)):
        torch.manual_seed(42)
        embs = torch.normal(mean=0, std=1, size=(vocab_size, dim))
        embs /= embs.norm(dim=-1, keepdim=True)
        idx_1, idx_2 = 400, 600
        logits = (0.5 * embs[idx_1] + 0.5 * embs[idx_2]) @ embs.T
        probs = logits.softmax(dim=-1)
        median = probs.median()
        probs, idcs = probs.topk(10, sorted=True)
        gap_diff = probs[1] - probs[2]
        gap_ratio = probs[1] / probs[2]
        df.append(
            {
                "dim": dim,
                "vocab": vocab_size,
                "gap_diff": gap_diff.item(),
                "gap_ratio": gap_ratio.item(),
                "top1": probs[0].item(),
                "top2": probs[1].item(),
                "top3": probs[2].item(),
                "median": median.item()
            } 
        )
df = pl.DataFrame(df)

In [ ]:
plotly.express.line(
    df,
    x="dim",
    y="gap_ratio",
    color="vocab",
    markers=True,
    title="Probability ratio between the correct tokens and most probably incorrect token"
)

In [ ]:
plotly.express.line(
    df,
    x="vocab",
    y="gap_ratio",
    color="dim",
    markers=True
)
    

In [ ]:
plotly.express.line(
    df,
    x="dim",
    y="gap_diff",
    color="vocab",
    markers=True,
    title="Difference in probability between the correct tokens and most probably incorrect token"
)
    

In [ ]:
plotly.express.line(
    df,
    x="vocab",
    y="gap_diff",
    color="dim",
    markers=True
)
    

In [ ]:
for dim in [100, 1000, 10000]:
    for vocab_size in [1000, 10000, 100000]:
        torch.manual_seed(1)
        embs = torch.normal(mean=0, std=1, size=(vocab_size, dim))
        embs /= embs.norm(dim=-1, keepdim=True)
        idx_1, idx_2 = 400, 600
        logits = (0.5 * embs[idx_1] + 0.5 * embs[idx_2]) @ embs.T
        probs = logits.softmax(dim=-1)

        probs, idcs = probs.topk(10, sorted=True)
        plotly.express.bar(y=probs, x=list(map(str, idcs.tolist())), title=f"Random Normalized Embs topk (vocab={vocab_size}, dim={dim})", text_auto=".6f").show()


In [ ]:
import transformers
model_ckpt = "meta-llama/Llama-3.2-1B"
model = transformers.AutoModel.from_pretrained(model_ckpt).eval()

for vocab_size in [1000, 100000]:
    torch.manual_seed(0)
    embs = model.embed_tokens.weight[torch.randint(0, model.config.vocab_size, (vocab_size,))].detach()
    idx_1, idx_2 = 400, 600
    logits = (0.5 * embs[idx_1] + 0.5 * embs[idx_2]) @ embs.T
    probs = logits.softmax(dim=-1)
    probs, idcs = probs.topk(10, sorted=True)
    plotly.express.bar(y=probs, x=list(map(str, idcs.tolist())), title=f"LLama 1B - topk (vocab_size={vocab_size}, dim={embs.shape[-1]})", text_auto=".6f").show()

In [ ]:
import transformers
model_ckpt = "meta-llama/Llama-3.1-70B"
model = transformers.AutoModel.from_pretrained(model_ckpt).eval()

for vocab_size in [1000, 100000]:
    torch.manual_seed(0)
    embs = model.embed_tokens.weight[torch.randint(0, model.config.vocab_size, (vocab_size,))].detach()
    idx_1, idx_2 = 400, 600
    logits = (0.5 * embs[idx_1] + 0.5 * embs[idx_2]) @ embs.T
    probs = logits.softmax(dim=-1)
    probs, idcs = probs.topk(10, sorted=True)
    plotly.express.bar(y=probs, x=list(map(str, idcs.tolist())), title=f"LLama 70B - topk (vocab_size={vocab_size}, dim={embs.shape[-1]})", text_auto=".6f").show()